# **Movie Recommendation System with Content-Based Filtering**

---

## Problem Statement

With thousands of movies available on streaming platforms, users struggle to discover content matching their personal taste. This project builds a **content-based recommendation system** that suggests movies based on genre similarity and individual user preferences — no cross-user interaction data required.

## Objectives

1. Item-to-item recommendations using **Cosine Similarity**
2. Single-user personalized recommendations weighted by watch history ratings
3. Multi-user personalized recommendations with individual preference profiles

# **Dataset Overview**

**Dataset:** [Top Rated TMDB Movies — 10K](https://www.kaggle.com/datasets/ahsanaseer/top-rated-tmdb-movies-10k)

| Column | Type | Description |
|---|---|---|
| `id` | int | Unique movie id number |
| `title` | str | Movie title |
| `genre` | str | Movie genre tags |
| `original_language` | str | Original language in which the movie released |
| `overview` | str | Summary of the movie |
| `popularity` | float | Movie popularity |
| `release_date` | str | Movie release date |
| `vote_average` | float | Average user rating on a 0–10 scale |
| `vote_count` | int | Total number of user ratings |

> <br>`vote_average` serves as a **preference weight** in the user feature vector — genres from higher-rated movies contribute more to the user's profile, reflecting stronger affinity.

# **Import Library**

In [1]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)

# **Load Dataset**

In [35]:
df = pd.read_csv("../dataset/movies.csv")
df.head()

,id,title,genre,original_language,overview,popularity,release_date,vote_average,vote_count
0,278,The Shawshank Redemption,"Drama,Crime",en,Framed in the 1940s for the double murder of h...,94.075,1994-09-23,8.7,21862
1,19404,Dilwale Dulhania Le Jayenge,"Comedy,Drama,Romance",hi,"Raj is a rich, carefree, happy-go-lucky second...",25.408,1995-10-19,8.7,3731
2,238,The Godfather,"Drama,Crime",en,"Spanning the years 1945 to 1955, a chronicle o...",90.585,1972-03-14,8.7,16280
3,424,Schindler's List,"Drama,History,War",en,The true story of how businessman Oskar Schind...,44.761,1993-12-15,8.6,12959
4,240,The Godfather: Part II,"Drama,Crime",en,In the continuing saga of the Corleone crime f...,57.749,1974-12-20,8.6,9811


# **Data Cleaning**

In [36]:
# Summarize each column's dtype, null count, and cardinality in one view
df_item = []

for col in df.columns:
    df_item.append([
        col, 
        df[col].dtype, 
        df[col].isna().sum(), 
        round(df[col].isna().sum()/len(df[col])*100, 2), 
        df[col].nunique()
    ])

df_overview = pd.DataFrame(
    columns=["column_name", "data_type", "null", "null_percentage", "unique"], 
    data=df_item
)

print(f"Number of rows and columns : {df.shape}")
df_overview

Number of rows and columns : (10000, 9)


,column_name,data_type,null,null_percentage,unique
0,id,int64,0,0.00,10000
1,title,str,0,0.00,9661
2,genre,str,3,0.03,2123
3,original_language,str,0,0.00,43
4,overview,str,13,0.13,9985
5,popularity,float64,0,0.00,8511
6,release_date,str,0,0.00,6113
7,vote_average,float64,0,0.00,42
8,vote_count,int64,0,0.00,3191


## Missing Value

In [37]:
# Drop rows with missing genre or overview
df.dropna(subset=["genre", "overview"], inplace=True)

df.isna().sum()

id                   0
title                0
genre                0
original_language    0
overview             0
popularity           0
release_date         0
vote_average         0
vote_count           0
dtype: int64

## Duplicated Data

In [38]:
# Check duplicated data
print(f"Number of duplicate rows : {df.duplicated().sum()} ({df.duplicated().sum()/len(df)}%)")

Number of duplicate rows : 0 (0.0%)


## Inconsistent Data

In [39]:
# Convert release_date from string to datetime
df["release_date"] = pd.to_datetime(df["release_date"])

print("Data types for each column after conversion :")
print(df.dtypes)

Data types for each column after conversion :
id                            int64
title                           str
genre                           str
original_language               str
overview                        str
popularity                  float64
release_date         datetime64[us]
vote_average                float64
vote_count                    int64
dtype: object


## Dataset After Cleaning

In [40]:
# Dataset overview after cleaning
df_item = []

for col in df.columns:
    df_item.append([
        col, 
        df[col].dtype, 
        df[col].isna().sum(), 
        round(df[col].isna().sum()/len(df[col])*100, 2), 
        df[col].nunique()
    ])

df_overview = pd.DataFrame(
    columns=["column_name", "data_type", "null", "null_percentage", "unique"], 
    data=df_item
)

print(f"Number of rows and columns : {df.shape}")
df_overview

Number of rows and columns : (9985, 9)


,column_name,data_type,null,null_percentage,unique
0,id,int64,0,0.0,9985
1,title,str,0,0.0,9646
2,genre,str,0,0.0,2123
3,original_language,str,0,0.0,43
4,overview,str,0,0.0,9983
5,popularity,float64,0,0.0,8497
6,release_date,datetime64[us],0,0.0,6107
7,vote_average,float64,0,0.0,42
8,vote_count,int64,0,0.0,3191


In [41]:
# Cleaned dataset
df.reset_index(drop=True, inplace=True)
df

,id,title,genre,original_language,overview,popularity,release_date,vote_average,vote_count
0,278,The Shawshank Redemption,"Drama,Crime",en,Framed in the 1940s for the double murder of h...,94.075,1994-09-23,8.7,21862
1,19404,Dilwale Dulhania Le Jayenge,"Comedy,Drama,Romance",hi,"Raj is a rich, carefree, happy-go-lucky second...",25.408,1995-10-19,8.7,3731
2,238,The Godfather,"Drama,Crime",en,"Spanning the years 1945 to 1955, a chronicle o...",90.585,1972-03-14,8.7,16280
3,424,Schindler's List,"Drama,History,War",en,The true story of how businessman Oskar Schind...,44.761,1993-12-15,8.6,12959
4,240,The Godfather: Part II,"Drama,Crime",en,In the continuing saga of the Corleone crime f...,57.749,1974-12-20,8.6,9811
...,...,...,...,...,...,...,...,...,...
9980,10196,The Last Airbender,"Action,Adventure,Fantasy",en,"The story follows the adventures of Aang, a yo...",98.322,2010-06-30,4.7,3347
9981,331446,Sharknado 3: Oh Hell No!,"Action,TV Movie,Science Fiction,Comedy,Adventure",en,The sharks take bite out of the East Coast whe...,12.490,2015-07-22,4.7,417
9982,13995,Captain America,"Action,Science Fiction,War",en,"During World War II, a brave, patriotic Americ...",18.333,1990-12-14,4.6,332
9983,2312,In the Name of the King: A Dungeon Siege Tale,"Adventure,Fantasy,Action,Drama",en,A man named Farmer sets out to rescue his kidn...,15.159,2007-11-29,4.7,668


# **Data Preprocessing**

Keep only columns needed for the recommendation pipeline:
- **id/title**: item identifiers for lookup and display
- **genre**: the feature used to compute similarity
- **vote_average**: used as a preference weight when building user profiles

In [42]:
# Select columns needed for the recommendation
df = df[["id", "title", "genre", "vote_average"]]
df

,id,title,genre,vote_average
0,278,The Shawshank Redemption,"Drama,Crime",8.7
1,19404,Dilwale Dulhania Le Jayenge,"Comedy,Drama,Romance",8.7
2,238,The Godfather,"Drama,Crime",8.7
3,424,Schindler's List,"Drama,History,War",8.6
4,240,The Godfather: Part II,"Drama,Crime",8.6
...,...,...,...,...
9980,10196,The Last Airbender,"Action,Adventure,Fantasy",4.7
9981,331446,Sharknado 3: Oh Hell No!,"Action,TV Movie,Science Fiction,Comedy,Adventure",4.7
9982,13995,Captain America,"Action,Science Fiction,War",4.6
9983,2312,In the Name of the King: A Dungeon Siege Tale,"Adventure,Fantasy,Action,Drama",4.7


# **Create Item-Feature Matrix**

In [ ]:
# Convert genre strings into binary vectors using CountVectorizer with a comma-based tokenization
vect = CountVectorizer(tokenizer=lambda x: x.split(","))

df_genre_vect = vect.fit_transform(df["genre"])

df_genre_vect = pd.DataFrame(
    df_genre_vect.toarray(), 
    columns=vect.get_feature_names_out()
)
df_genre_vect = pd.concat([df[["id", "title", "vote_average"]], df_genre_vect], axis=1)
df_genre_vect

,id,title,vote_average,action,adventure,animation,comedy,crime,drama,family,fantasy,history,horror,music,mystery,romance,science fiction,thriller,tv movie,war,western
0,278,The Shawshank Redemption,8.7,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0
1,19404,Dilwale Dulhania Le Jayenge,8.7,0,0,0,1,0,1,0,0,0,0,0,0,1,0,0,0,0,0
2,238,The Godfather,8.7,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0
3,424,Schindler's List,8.6,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,1,0
4,240,The Godfather: Part II,8.6,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9980,10196,The Last Airbender,4.7,1,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
9981,331446,Sharknado 3: Oh Hell No!,4.7,1,1,0,1,0,0,0,0,0,0,0,0,0,1,0,1,0,0
9982,13995,Captain America,4.6,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0
9983,2312,In the Name of the King: A Dungeon Siege Tale,4.7,1,1,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0


# **Item-to-Item Recommendation**

The most direct form of content-based filtering: given a movie a user just watched, surface the 10 most genre-similar titles in the catalog.

Cosine similarity measures the angle between two genre vectors — score of **1.0** means identical genres, **0.0** means no overlap. When multiple movies share a perfect score, `vote_average` is used as a tiebreaker.

In [44]:
# Select watched movies
df_genre_vect[df_genre_vect["title"] == "The Godfather"]

,id,title,vote_average,action,adventure,animation,comedy,crime,drama,family,fantasy,history,horror,music,mystery,romance,science fiction,thriller,tv movie,war,western
2,238,The Godfather,8.7,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0


In [45]:
# Select genre columns as features for similarity computation
movies_matrix = df_genre_vect[df_genre_vect["title"] == "The Godfather"].loc[:, "action":]
movies_matrix

,action,adventure,animation,comedy,crime,drama,family,fantasy,history,horror,music,mystery,romance,science fiction,thriller,tv movie,war,western
2,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0


In [46]:
# Calculate cosine_similarity score, returns a (1, N) 2D array — one row per reference movie
cosine_score = cosine_similarity(movies_matrix, df_genre_vect.loc[:, "action":])

df_genre_vect["cosine_score"] = cosine_score.reshape(-1)
df_genre_vect.head()

,id,title,vote_average,action,adventure,animation,comedy,crime,drama,family,fantasy,history,horror,music,mystery,romance,science fiction,thriller,tv movie,war,western,cosine_score
0,278,The Shawshank Redemption,8.7,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,1.000000
1,19404,Dilwale Dulhania Le Jayenge,8.7,0,0,0,1,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0.408248
2,238,The Godfather,8.7,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,1.000000
3,424,Schindler's List,8.6,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,1,0,0.408248
4,240,The Godfather: Part II,8.6,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,1.000000


In [47]:
# Sort and exclude the reference movie — cosine similarity of a movie with itself is always 1.0
top10_movies = df_genre_vect[df_genre_vect["title"] != "The Godfather"].sort_values(["cosine_score", "vote_average"], ascending=[False, False])[["title", "vote_average", "cosine_score"]]
top10_movies.head(10)

,title,vote_average,cosine_score
0,The Shawshank Redemption,8.7,1.0
4,The Godfather: Part II,8.6,1.0
23,GoodFellas,8.5,1.0
26,Once Upon a Time in America,8.5,1.0
36,City of God,8.4,1.0
128,The Hate U Give,8.2,1.0
148,Taxi Driver,8.2,1.0
209,"Three Billboards Outside Ebbing, Missouri",8.1,1.0
233,Rocco and His Brothers,8.1,1.0
264,To Kill a Mockingbird,8.0,1.0


All top results share the exact genre combination **Drama + Crime** (cosine score = 1.0), which is identical to *The Godfather*. They are ranked by `vote_average` to surface the most acclaimed matches first.

The high number of perfect scores reflects a fundamental limitation of binary encoding — a film that is 90% drama and 10% crime scores identically to one where both genres are equally prominent. TF-IDF weighting or NLP embeddings from `overview` would create more discriminative vectors.

# **Single-User Personalization**

Pure similarity matching treats all users the same. A more personal approach builds an individual genre preference profile from a user's watch history, then scores every unseen movie against that profile.

Each genre is **weighted by `vote_average`** — genres from higher-rated movies contribute more to the profile. The vector is then normalized to sum to 1.

In [48]:
# User's watched movies
list_watched_movies = ["Avengers: Infinity War", "How to Train Your Dragon: Homecoming", "Jujutsu Kaisen 0"]
list_watched_movies

['Avengers: Infinity War',
 'How to Train Your Dragon: Homecoming',
 'Jujutsu Kaisen 0']

In [49]:
# Filter item-feature matrix to only the watched movies
df_watched_movies = df_genre_vect[df_genre_vect["title"].isin(list_watched_movies)]
df_watched_movies = df_watched_movies.drop(columns="cosine_score")
df_watched_movies

,id,title,vote_average,action,adventure,animation,comedy,crime,drama,family,fantasy,history,horror,music,mystery,romance,science fiction,thriller,tv movie,war,western
96,299536,Avengers: Infinity War,8.3,1,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
188,638507,How to Train Your Dragon: Homecoming,8.1,1,1,1,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0
556,810693,Jujutsu Kaisen 0,7.8,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0


In [50]:
# Weight genres by vote average to build the user's profile
df_item_feature_pop = df_watched_movies.loc[:, "action":].multiply(df_watched_movies["vote_average"], axis=0)
df_item_feature_pop

,action,adventure,animation,comedy,crime,drama,family,fantasy,history,horror,music,mystery,romance,science fiction,thriller,tv movie,war,western
96,8.3,8.3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,8.3,0.0,0.0,0.0,0.0
188,8.1,8.1,8.1,0.0,0.0,0.0,8.1,8.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
556,7.8,0.0,7.8,0.0,0.0,0.0,0.0,7.8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [51]:
# Normalize to a probability distribution summing to 1
user_feature_vector = df_item_feature_pop.sum() / df_item_feature_pop.sum().sum()
user_feature_vector

action             0.272523
adventure          0.184685
animation          0.179054
comedy             0.000000
crime              0.000000
drama              0.000000
family             0.091216
fantasy            0.179054
history            0.000000
horror             0.000000
music              0.000000
mystery            0.000000
romance            0.000000
science fiction    0.093468
thriller           0.000000
tv movie           0.000000
war                0.000000
western            0.000000
dtype: float64

## User Genre Profile

| Genre | Weight | Source |
|---|---|---|
| **Action** | 27.3% | Present in all 3 watched movies |
| **Adventure** | 18.5% | Avengers + How to Train Your Dragon |
| **Animation** | 17.9% | How to Train Your Dragon + Jujutsu Kaisen |
| **Fantasy** | 17.9% | How to Train Your Dragon + Jujutsu Kaisen |
| **Science Fiction** | 9.3% | Avengers only |
| **Family** | 9.1% | How to Train Your Dragon only |

This vector is used as genre importance weights when scoring unseen movies. A movie covering all six non-zero genres (**Action + Adventure + Animation + Fantasy + Family + Science Fiction**) scores a perfect 1.0.

In [52]:
# Exclude already-watched movies to avoid recommending content the user has already seen
df_unwatched = df_genre_vect[~df_genre_vect["title"].isin(list_watched_movies)]
df_unwatched = df_unwatched.drop(columns="cosine_score")
df_unwatched = df_unwatched.reset_index(drop=True)
df_unwatched

,id,title,vote_average,action,adventure,animation,comedy,crime,drama,family,fantasy,history,horror,music,mystery,romance,science fiction,thriller,tv movie,war,western
0,278,The Shawshank Redemption,8.7,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0
1,19404,Dilwale Dulhania Le Jayenge,8.7,0,0,0,1,0,1,0,0,0,0,0,0,1,0,0,0,0,0
2,238,The Godfather,8.7,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0
3,424,Schindler's List,8.6,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,1,0
4,240,The Godfather: Part II,8.6,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9977,10196,The Last Airbender,4.7,1,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
9978,331446,Sharknado 3: Oh Hell No!,4.7,1,1,0,1,0,0,0,0,0,0,0,0,0,1,0,1,0,0
9979,13995,Captain America,4.6,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0
9980,2312,In the Name of the King: A Dungeon Siege Tale,4.7,1,1,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0


In [53]:
# Multiply each movie's genre vector by the user's genre weights
df_recom = df_unwatched.loc[:, "action":].multiply(user_feature_vector, axis=1)
df_recom = pd.concat([df_unwatched[["title", "vote_average"]], df_recom], axis=1)
df_recom

,title,vote_average,action,adventure,animation,comedy,crime,drama,family,fantasy,history,horror,music,mystery,romance,science fiction,thriller,tv movie,war,western
0,The Shawshank Redemption,8.7,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
1,Dilwale Dulhania Le Jayenge,8.7,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
2,The Godfather,8.7,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
3,Schindler's List,8.6,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
4,The Godfather: Part II,8.6,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9977,The Last Airbender,4.7,0.272523,0.184685,0.0,0.0,0.0,0.0,0.0,0.179054,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
9978,Sharknado 3: Oh Hell No!,4.7,0.272523,0.184685,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.093468,0.0,0.0,0.0,0.0
9979,Captain America,4.6,0.272523,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.093468,0.0,0.0,0.0,0.0
9980,In the Name of the King: A Dungeon Siege Tale,4.7,0.272523,0.184685,0.0,0.0,0.0,0.0,0.0,0.179054,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0


In [54]:
# Sum across all genres to get a single relevance score per movie
df_recom["total_score"] = df_recom.loc[:, "action":].sum(axis=1)
df_recom

,title,vote_average,action,adventure,animation,comedy,crime,drama,family,fantasy,history,horror,music,mystery,romance,science fiction,thriller,tv movie,war,western,total_score
0,The Shawshank Redemption,8.7,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000
1,Dilwale Dulhania Le Jayenge,8.7,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000
2,The Godfather,8.7,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000
3,Schindler's List,8.6,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000
4,The Godfather: Part II,8.6,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9977,The Last Airbender,4.7,0.272523,0.184685,0.0,0.0,0.0,0.0,0.0,0.179054,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.636261
9978,Sharknado 3: Oh Hell No!,4.7,0.272523,0.184685,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.093468,0.0,0.0,0.0,0.0,0.550676
9979,Captain America,4.6,0.272523,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.093468,0.0,0.0,0.0,0.0,0.365991
9980,In the Name of the King: A Dungeon Siege Tale,4.7,0.272523,0.184685,0.0,0.0,0.0,0.0,0.0,0.179054,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.636261


In [55]:
# Get the top 10 recommendations by total score
top10_movies_one_user = df_recom[["title", "total_score"]].sort_values("total_score", ascending=False).head(10)
top10_movies_one_user

,title,total_score
3860,Pokémon: The Rise of Darkrai,1.000000
5468,Pokémon: Jirachi - Wish Maker,1.000000
3612,Pokémon: Lucario and the Mystery of Mew,1.000000
7017,Final Fantasy: The Spirits Within,0.908784
4023,Dragon Ball: Curse of the Blood Rubies,0.908784
2720,Final Fantasy VII: Advent Children,0.908784
74,Justice League Dark: Apokolips War,0.908784
5778,Godzilla: The Planet Eater,0.908784
3512,Green Lantern: First Flight,0.908784
2147,Wonder Woman,0.908784


**Pokémon films dominate** the top 3 (score = 1.0) because they cover all **six** of the user's non-zero genres: Action, Adventure, Animation, Fantasy, Family, and Science Fiction — summing to a perfect score of 1.0. The scoring is a weighted dot product — movies matching more of the user's preferred genres accumulate higher scores.

## Encapsulated Pipeline

The step-by-step process above is wrapped into a single reusable function for cleaner usage.

In [ ]:
def movie_recommendations(list_watched_movies):
    df = pd.read_csv("../dataset/movies.csv")
    df = df.dropna(subset=["genre", "overview"]).reset_index(drop=True)
    df = df[["id", "title", "genre", "vote_average"]]

    vect = CountVectorizer(tokenizer=lambda x: x.split(","))
    df_genre_vect = vect.fit_transform(df["genre"])
    
    df_genre_vect = pd.DataFrame(
        df_genre_vect.toarray(), 
        columns=vect.get_feature_names_out()
    )
    df_genre_vect = pd.concat([df[["id", "title", "vote_average"]], df_genre_vect], axis=1)

    df_watched = df_genre_vect[df_genre_vect["title"].isin(list_watched_movies)]

    # Weight genres by vote average to build the user's profile
    df_item_feature_pop = df_watched.loc[:, "action":].multiply(df_watched["vote_average"], axis=0)

    # Normalize to a probability distribution
    user_feature_vector = df_item_feature_pop.sum() / df_item_feature_pop.sum().sum()

    df_unwatched = df_genre_vect[~df_genre_vect["title"].isin(list_watched_movies)].reset_index(drop=True)

    df_recom = df_unwatched.loc[:, "action":].multiply(user_feature_vector, axis=1)
    df_recom = pd.concat([df_unwatched[["title", "vote_average"]], df_recom], axis=1)
    df_recom["total_score"] = df_recom.loc[:, "action":].sum(axis=1)

    return df_recom[["title", "total_score"]].sort_values("total_score", ascending=False).head(10)

In [57]:
# Check function movies_recommendation
movie_recommendations(list_watched_movies)

,title,total_score
3860,Pokémon: The Rise of Darkrai,1.000000
5468,Pokémon: Jirachi - Wish Maker,1.000000
3612,Pokémon: Lucario and the Mystery of Mew,1.000000
7017,Final Fantasy: The Spirits Within,0.908784
4023,Dragon Ball: Curse of the Blood Rubies,0.908784
2720,Final Fantasy VII: Advent Children,0.908784
74,Justice League Dark: Apokolips War,0.908784
5778,Godzilla: The Planet Eater,0.908784
3512,Green Lantern: First Flight,0.908784
2147,Wonder Woman,0.908784


# **Multi-User Personalization**

The single-user approach scales naturally to any number of users. Each user here brings a distinct watch history reflecting a different taste profile — the system builds an independent preference vector for each one and generates personalized recommendations accordingly.

| User | Persona | Movies Watched |
|---|---|---|
| **User 1** | Action / Sci-Fi | Avengers: Infinity War, Captain America, The Last Airbender |
| **User 2** | Historical Drama | Schindler's List, The Godfather: Part II, The Shawshank Redemption |
| **User 3** | Animation / Fantasy | How to Train Your Dragon: Homecoming, Jujutsu Kaisen 0, The Last Airbender |
| **User 4** | Crime / Thriller | We Own the Night, The Godfather, The Godfather: Part II |


In [72]:
list_watched_moviesId = [299536, 13995, 10196, 424, 240, 278, 638507, 810693, 2001, 238]

# User-Item Rating Matrix: rows = users, columns = movie IDs, values = ratings (1–10)
df_user_rating = pd.DataFrame({
    "user": ["user 1", "user 2", "user 3", "user 4"],
    299536: [9, None, None, None],
    13995: [8, None, None, None],
    10196: [7, None, 7, None],
    424: [None, 9, None, None],
    240: [None, 8, None, 7],
    278: [None, 7, None, None],
    638507: [None, None, 9, None],
    810693: [None, None, 8, None],
    2001: [None, None, None, 9],
    238: [None, None, None, 8]
})
df_user_rating

,user,299536,13995,10196,424,240,278,638507,810693,2001,238
0,user 1,9.0,8.0,7.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,user 2,NaN,NaN,NaN,9.0,8.0,7.0,NaN,NaN,NaN,NaN
2,user 3,NaN,NaN,7.0,NaN,NaN,NaN,9.0,8.0,NaN,NaN
3,user 4,NaN,NaN,NaN,NaN,7.0,NaN,NaN,NaN,9.0,8.0


In [73]:
# Extract item-feature matrix for all movies in the combined watch history
df_item_feature_multiple = df_genre_vect[df_genre_vect["id"].isin(list_watched_moviesId)]
df_item_feature_multiple = df_item_feature_multiple.drop(columns="cosine_score")
df_item_feature_multiple

,id,title,vote_average,action,adventure,animation,comedy,crime,drama,family,fantasy,history,horror,music,mystery,romance,science fiction,thriller,tv movie,war,western
0,278,The Shawshank Redemption,8.7,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0
2,238,The Godfather,8.7,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0
3,424,Schindler's List,8.6,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,1,0
4,240,The Godfather: Part II,8.6,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0
96,299536,Avengers: Infinity War,8.3,1,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
188,638507,How to Train Your Dragon: Homecoming,8.1,1,1,1,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0
556,810693,Jujutsu Kaisen 0,7.8,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
4722,2001,We Own the Night,6.7,0,0,0,0,1,1,0,0,0,0,0,0,0,0,1,0,0,0
9980,10196,The Last Airbender,4.7,1,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
9982,13995,Captain America,4.6,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0


In [74]:
# Reindex rows to match the column order of df_user_rating
df_item_feature_multiple = df_item_feature_multiple.set_index("id")
df_item_feature_multiple = df_item_feature_multiple.loc[list(df_user_rating.columns[1:]), :]
df_item_feature_multiple

,title,vote_average,action,adventure,animation,comedy,crime,drama,family,fantasy,history,horror,music,mystery,romance,science fiction,thriller,tv movie,war,western
id,,,,,,,,,,,,,,,,,,,,
299536,Avengers: Infinity War,8.3,1,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
13995,Captain America,4.6,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0
10196,The Last Airbender,4.7,1,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
424,Schindler's List,8.6,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,1,0
240,The Godfather: Part II,8.6,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0
278,The Shawshank Redemption,8.7,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0
638507,How to Train Your Dragon: Homecoming,8.1,1,1,1,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0
810693,Jujutsu Kaisen 0,7.8,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
2001,We Own the Night,6.7,0,0,0,0,1,1,0,0,0,0,0,0,0,0,1,0,0,0


In [75]:
list_user_feature_vector = []

for user in range(len(df_user_rating)):
    # Weight each genre by the user's rating
    df_item_feature_rating = df_item_feature_multiple.loc[list(df_user_rating.columns[1:]), "action":"western"].multiply(df_user_rating.iloc[user, 1:].fillna(0), axis=0)

    # Normalize so all user vectors sum to 1
    user_feature_vector = df_item_feature_rating.sum() / df_item_feature_rating.sum().sum()
    list_user_feature_vector.append(user_feature_vector.values)

# Stack individual vectors into a matrix: rows = users, columns = genres
user_feature_matrix = pd.DataFrame(
    list_user_feature_vector, 
    columns=df_item_feature_rating.columns, 
    index=df_user_rating["user"]
)
user_feature_matrix

,action,adventure,animation,comedy,crime,drama,family,fantasy,history,horror,music,mystery,romance,science fiction,thriller,tv movie,war,western
user,,,,,,,,,,,,,,,,,,
user 1,0.333333,0.222222,0.000000,0.0,0.000000,0.000000,0.0,0.097222,0.000000,0.0,0.0,0.0,0.0,0.236111,0.000000,0.0,0.111111,0.0
user 2,0.000000,0.000000,0.000000,0.0,0.263158,0.421053,0.0,0.000000,0.157895,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.157895,0.0
user 3,0.266667,0.177778,0.188889,0.0,0.000000,0.000000,0.1,0.266667,0.000000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.0
user 4,0.000000,0.000000,0.000000,0.0,0.421053,0.421053,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,0.157895,0.0,0.000000,0.0


## User Profile Comparison

Each user's distinct watch history produces a clearly different genre preference profile:

| Genre | User 1 | User 2 | User 3 | User 4 |
|---|---|---|---|---|
| **Action** | 33.3% | — | 26.7% | — |
| **Drama** | — | 42.1% | — | 42.1% |
| **Crime** | — | 26.3% | — | 42.1% |
| **Science Fiction** | 23.6% | — | — | — |
| **Adventure** | 22.2% | — | 17.8% | — |
| **Fantasy** | 9.7% | — | 26.7% | — |
| **Animation** | — | — | 18.9% | — |
| **History** | — | 15.8% | — | — |
| **War** | 11.1% | 15.8% | — | — |
| **Family** | — | — | 10.0% | — |
| **Thriller** | — | — | — | 15.8% |

- **User 1** is dominated by Action + Sci-Fi (Avengers and Captain America drive both genres) → expects superhero and sci-fi films
- **User 2** is Drama-heavy with unique History/War signal from Schindler's List → expects historical and prestige dramas
- **User 3** has Action and Fantasy equally weighted, with the strongest Animation signal of all users → expects animated fantasy and anime films
- **User 4** has Crime and Drama equally dominant, with Thriller as the differentiator → expects crime thriller films

In [ ]:
# Use the full catalog for scoring — per-user exclusion is handled at display time
df_all_movies = df_genre_vect.drop(columns="cosine_score")
df_all_movies = df_all_movies.reset_index(drop=True)
df_all_movies

,id,title,vote_average,action,adventure,animation,comedy,crime,drama,family,fantasy,history,horror,music,mystery,romance,science fiction,thriller,tv movie,war,western
0,278,The Shawshank Redemption,8.7,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0
1,19404,Dilwale Dulhania Le Jayenge,8.7,0,0,0,1,0,1,0,0,0,0,0,0,1,0,0,0,0,0
2,238,The Godfather,8.7,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0
3,424,Schindler's List,8.6,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,1,0
4,240,The Godfather: Part II,8.6,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9980,10196,The Last Airbender,4.7,1,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
9981,331446,Sharknado 3: Oh Hell No!,4.7,1,1,0,1,0,0,0,0,0,0,0,0,0,1,0,1,0,0
9982,13995,Captain America,4.6,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0
9983,2312,In the Name of the King: A Dungeon Siege Tale,4.7,1,1,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0


In [77]:
list_movie_score = []

for user in range(len(user_feature_matrix)):
    # Dot product: genre presence × user weights, summed per movie
    movie_score = df_all_movies.loc[:, "action":"western"].multiply(user_feature_matrix.iloc[user]).sum(axis=1)
    list_movie_score.append(movie_score.values)

user_score_matrix = pd.DataFrame(
    list_movie_score, 
    columns=df_all_movies["title"].values, 
    index=user_feature_matrix.index
)
user_score_matrix

,The Shawshank Redemption,Dilwale Dulhania Le Jayenge,The Godfather,Schindler's List,The Godfather: Part II,Impossible Things,Spirited Away,Your Eyes Tell,Dou kyu sei – Classmates,Your Name.,12 Angry Men,Gabriel's Inferno,Parasite,The Green Mile,Gabriel's Inferno: Part II,The Dark Knight,"The Good, the Bad and the Ugly",Pulp Fiction,The Lord of the Rings: The Return of the King,Gabriel's Inferno: Part III,Forrest Gump,Cinema Paradiso,Seven Samurai,GoodFellas,Violet Evergarden: The Movie,Life Is Beautiful,Once Upon a Time in America,Harakiri,Psycho,"Josee, the Tiger and the Fish",A Dog's Will,Grave of the Fireflies,One Flew Over the Cuckoo's Nest,Fight Club,Evangelion: 3.0+1.0 Thrice Upon a Time,Spider-Man: Into the Spider-Verse,City of God,Hope,A Silent Voice: The Movie,Hotarubi no Mori e,Howl's Moving Castle,Neon Genesis Evangelion: The End of Evangelion,The Empire Strikes Back,Sunset Boulevard,Wolfwalkers,The Lord of the Rings: The Fellowship of the Ring,Ikiru,Primal: Tales of Savagery,The Pianist,Interstellar,Whiplash,The Lord of the Rings: The Two Towers,American History X,Rear Window,Inception,Demon Slayer -Kimetsu no Yaiba- The Movie: Mugen Train,Given,City Lights,High and Low,The Great Dictator,Se7en,Top Gun: Maverick,Princess Mononoke,Dedicated to my ex,The Silence of the Lambs,Come and See,Clouds,Dead Poets Society,Modern Times,Five Feet Apart,Léon: The Professional,Once Upon a Time in the West,Hamilton,Paths of Glory,Justice League Dark: Apokolips War,Back to the Future,My Hero Academia: Heroes Rising,Life in a Year,Better Days,Woman in the Dunes,We All Loved Each Other So Much,Tokyo Story,Perfect Blue,The Legend of 1900,Apocalypse Now,Rascal Does Not Dream of a Dreaming Girl,Avengers: Endgame,Le Trou,Miracle in Cell No. 7,Mommy,Oldboy,Steven Universe: The Movie,The Art of Racing in the Rain,Zack Snyder's Justice League,Klaus,The Handmaiden,Avengers: Infinity War,The Intouchables,KonoSuba: God's Blessing on this Wonderful World! Legend of Crimson,Everything Everywhere All at Once,The Lion King,It's a Wonderful Life,Persona,Black Beauty,Bicycle Thieves,Wolf Children,Maquia: When the Promised Flower Blooms,Il Sorpasso,Green Book,Violet Evergarden: Eternity and the Auto Memory Doll,I Want to Eat Your Pancreas,Sansho the Bailiff,Witness for the Prosecution,"Miraculous World: New York, United HeroeZ",Mortal Kombat Legends: Scorpion's Revenge,New Gods: Nezha Reborn,Stalker,8½,The Kid,The Apartment,Bo Burnham: Inside,Coco,The Shining,Doctor Who: The Day of the Doctor,A Clockwork Orange,The Seventh Seal,Investigation of a Citizen Above Suspicion,Vertigo,The Hate U Give,Children of Paradise,Portrait of a Lady on Fire,Inglourious Basterds,Along with the Gods: The Last 49 Days,Call Me by Your Name,Star Wars,The Usual Suspects,Gladiator,Piper,The Prestige,Saving Private Ryan,Memento,Shutter Island,The Matrix,My Mom is a Character 3,Capernaum,The Help,Hacksaw Ridge,Soul,Taxi Driver,Wonder,Singin' in the Rain,Metropolis,Yojimbo,Joker,All About Eve,Scenes from a Marriage,Casablanca,La Dolce Vita,Bo Burnham: Make Happy,Andrei Rublev,The Father,Scarface,My Friends,The Departed,Rashomon,"Lock, Stock and Two Smoking Barrels",Ayla: The Daughter of War,Togo,Double Indemnity,Central Station,Sherlock Jr.,Me Against You: Mr. S's Vendetta,Out of the Clear Blue Sky,Nobody,Reservoir Dogs,Django Unchained,Far From the Tree,The Great War,Cruella,Full Metal Jacket,Good Will Hunting,Some Like It Hot,Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb,Wild Strawberries,Alien,Paper Lives,The Invisible Guest,"Paris, Texas",How to Train Your Dragon: Homecoming,Dersu Uzala,Pather Panchali,The Tale of The Princess Kaguya,Anne of Green Gables,Harry Potter and the Deathly Hallows: Part 2,Michael Jackson's Thriller,Big Deal on Madonna Street,The Truman Show,Red Beard,M,Believe Me: The Abduction of Lisa McVey,In the Mood for Love,Incendies,A Special Day,I corti,There Will Be Blood,Hannah Gadsby: Nanette,Eternal Sunshine of the Spotless Mind

## Recommendation for User 1

In [78]:
watched_id_user1 = df_user_rating.loc[df_user_rating["user"] == "user 1"].iloc[:, 1:].dropna(axis=1).columns.tolist()
watched_title_user1 = df_all_movies[df_all_movies["id"].isin(watched_id_user1)]["title"].tolist()

user_score_matrix.T[["user 1"]].drop(index=watched_title_user1, errors="ignore").sort_values("user 1", ascending=False).head(10)

user,user 1
The Postman,0.902778
The Shadow,0.888889
Tremors: A Cold Day in Hell,0.888889
Teenage Mutant Ninja Turtles III,0.888889
Final Fantasy: The Spirits Within,0.888889
Pokémon: The Rise of Darkrai,0.888889
Fantastic Four,0.888889
Sword Art Online: The Movie – Ordinal Scale,0.888889
Mystery Men,0.888889
The Wolverine,0.888889


## Recommendation for User 2

In [79]:
watched_id_user2 = df_user_rating.loc[df_user_rating["user"] == "user 2"].iloc[:, 1:].dropna(axis=1).columns.tolist()
watched_title_user2 = df_all_movies[df_all_movies["id"].isin(watched_id_user2)]["title"].tolist()

user_score_matrix.T[["user 2"]].drop(index=watched_title_user2, errors="ignore").sort_values("user 2", ascending=False).head(10)

user,user 2
Mississippi Burning,0.842105
BlacKkKlansman,0.842105
Gangs of New York,0.842105
Detroit,0.842105
Polytechnique,0.842105
Le Trou,0.842105
Les Misérables,0.842105
Papillon,0.842105
Richard Jewell,0.842105
A Twelve-Year Night,0.842105


## Recommendation for User 3

In [80]:
watched_id_user3 = df_user_rating.loc[df_user_rating["user"] == "user 3"].iloc[:, 1:].dropna(axis=1).columns.tolist()
watched_title_user3 = df_all_movies[df_all_movies["id"].isin(watched_id_user3)]["title"].tolist()

user_score_matrix.T[["user 3"]].drop(index=watched_title_user3, errors="ignore").sort_values("user 3", ascending=False).head(10)

user,user 3
Miraculous World: Shanghai – The Legend of Ladydragon,1.0
Pokemon the Movie: Mewtwo Strikes Back - Evolution,1.0
"Miraculous World: New York, United HeroeZ",1.0
NiNoKuni,1.0
Pokémon: Mewtwo Returns,1.0
Pokémon the Movie: Hoopa and the Clash of Ages,1.0
The Croods,1.0
Pokémon: Lucario and the Mystery of Mew,1.0
Pokémon: The First Movie,1.0
Raya and the Last Dragon,1.0


## Recommendation for User 4

In [81]:
watched_id_user4 = df_user_rating.loc[df_user_rating["user"] == "user 4"].iloc[:, 1:].dropna(axis=1).columns.tolist()
watched_title_user4 = df_all_movies[df_all_movies["id"].isin(watched_id_user4)]["title"].tolist()

user_score_matrix.T[["user 4"]].drop(index=watched_title_user4, errors="ignore").sort_values("user 4", ascending=False).head(10)

user,user 4
Bitch Slap,1.0
The Nile Hilton Incident,1.0
Whisper,1.0
The Dark Knight,1.0
Match Point,1.0
Blow Out,1.0
Trust,1.0
The Pelican Brief,1.0
Evil Under the Sun,1.0
Scarlet Street,1.0


## Result Comparison

The four users receive meaningfully different recommendations — a direct result of their distinct watch histories producing non-overlapping genre profiles.
- User 1 : Action/Sci-Fi profile surfaces superhero and space-themed films
- User 2 : Drama/History profile brings historical and prestige dramas to the top
- User 3 : Animation/Fantasy profile yields anime and animated features
- User 4 : Crime/Drama/Thriller profile surfaces crime and neo-noir films.

# **Conclusions**

## Summary

| Approach | Method | Input | Output |
|---|---|---|---|
| **Item-to-Item Recommendation** | Cosine Similarity | One reference movie | 10 genre-similar movies |
| **Single-User Personalization** | Weighted user vector | 3 watched movies | 10 personalized recommendations |
| **Multi-User Personalization** | Per-user weighted vectors | 4 users × distinct 3-movie histories | 10 recommendations per user |

## Key Findings

- Genre-based cosine similarity effectively clusters movies by content type, but binary encoding cannot distinguish degree of genre emphasis — many movies share a perfect 1.0 score
- Rating-weighted user vectors successfully capture individual preferences: genres from higher-rated movies contribute more to the profile, amplifying each user's strongest affinities
- Different watch histories produce non-overlapping genre profiles, resulting in genuinely distinct recommendation sets across users
- Per-user exclusion is critical — globally removing all watched movies across users would incorrectly disqualify valid candidates for users who never watched those films

## Limitations & Future Improvements

| Limitation | Improvement |
|---|---|
| Only genre used as feature | Add director, cast, or NLP embeddings from `overview` |
| Binary genre encoding | Use TF-IDF weighting |
| Scoring rewards genre count, not match quality | Use cosine similarity against the user vector |
| No feedback loop | Incorporate implicit signals (clicks, watch time) |
| Cold start for new users | Fall back to popularity-based recommendations |